# E6: Transformers with and without SMSGuard normalisation (run on Colab GPU)
Upload the `smsguard/` folder and `super_sms_dataset.csv` to Colab (or mount Drive). Runtime → Change runtime type → GPU.

In [ ]:
!pip -q install transformers datasets accelerate scikit-learn pandas

In [ ]:
import sys, random, numpy as np, pandas as pd, torch
sys.path.insert(0, '.')  # folder containing smsguard/
from smsguard.normalize import normalize
from smsguard.attacks import attack, ALL_ATTACKS
from smsguard.model import spam_keywords
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
SEED = 42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

In [ ]:
# Same loading and split as scripts/train.py, so results are comparable
df = pd.read_csv('super_sms_dataset.csv', encoding='cp1252', encoding_errors='replace')
df.columns = ['text', 'label']
df = df.dropna(); df['text'] = df.text.astype(str); df = df[df.text.str.strip() != '']
df['label'] = df.label.astype(int)
df = df.loc[~df.text.str.lower().str.split().str.join(' ').duplicated()].reset_index(drop=True)
X_tr, X_te, y_tr, y_te = train_test_split(df.text.tolist(), df.label.values, test_size=0.2, stratify=df.label.values, random_state=SEED)
# validation carved from TRAIN only (never use the test set for early stopping)
X_tr, X_va, y_tr, y_va = train_test_split(X_tr, y_tr, test_size=0.1, stratify=y_tr, random_state=SEED)
keywords = spam_keywords(X_tr, list(y_tr))
print(len(X_tr), len(X_va), len(X_te))

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback
from datasets import Dataset
MODEL = 'distilbert-base-uncased'   # also run 'roberta-base'
USE_NORM = True                     # run once with False for the ablation
ADV_TRAIN = False                   # run once with True
prep = (lambda t: normalize(t).text) if USE_NORM else (lambda t: t)
train_texts, train_labels = list(X_tr), list(y_tr)
if ADV_TRAIN:
    rng = random.Random(SEED)
    extra = [attack(t, rng.choice(ALL_ATTACKS), keywords, rate=rng.uniform(.15,.5), seed=SEED+i) for i,(t,l) in enumerate(zip(X_tr,y_tr)) if l==1]
    train_texts += extra; train_labels += [1]*len(extra)
tok = AutoTokenizer.from_pretrained(MODEL)
def ds(texts, labels):
    d = Dataset.from_dict({'text': [prep(t) for t in texts], 'label': [int(l) for l in labels]})
    return d.map(lambda b: tok(b['text'], truncation=True, max_length=96), batched=True)
train_ds, val_ds = ds(train_texts, train_labels), ds(X_va, y_va)

In [ ]:
def metrics(p):
    pred = p.predictions.argmax(-1); pr, rc, f1, _ = precision_recall_fscore_support(p.label_ids, pred, average='binary')
    return {'accuracy': accuracy_score(p.label_ids, pred), 'precision': pr, 'recall': rc, 'f1': f1}
model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2)
args = TrainingArguments('out', eval_strategy='epoch', save_strategy='epoch', num_train_epochs=4, learning_rate=2e-5,
    per_device_train_batch_size=32, per_device_eval_batch_size=64, load_best_model_at_end=True, metric_for_best_model='f1',
    fp16=torch.cuda.is_available(), seed=SEED, report_to='none')
trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds, tokenizer=tok, compute_metrics=metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)])
trainer.train()

In [ ]:
# Clean test set + every attack applied to all test spam (same protocol as scripts/train.py)
print('clean', trainer.evaluate(ds(X_te, y_te)))
spam_te = [t for t,l in zip(X_te,y_te) if l==1]
for a in ['none'] + ALL_ATTACKS:
    texts = spam_te if a=='none' else [attack(t, a, keywords, rate=0.3, seed=SEED) for t in spam_te]
    pred = trainer.predict(ds(texts, [1]*len(texts))).predictions.argmax(-1)
    print(f'{a:12s} detection rate = {pred.mean():.3f}')